In [1]:
# Direct wrappers
#
# https://tslearn.readthedocs.io/en/latest/gen_modules/tslearn.clustering.html#module-tslearn.clustering
# https://www.sktime.net/en/latest/api_reference/auto_generated/sktime.clustering.k_medoids.TimeSeriesKMedoids.html#sktime.clustering.k_medoids.TimeSeriesKMedoids


%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns

import random

# feature: addition of interpolation, smoothing and resolution selection
# feature: possibility to combine different clustering methods
# 
from timex.clustering import model_based_clustering # model-based clustering
from timex.clustering import distance_based_clustering # distance-based clustering
from timex.clustering import feature_based_clustering # feature-based clustering
from timex.clustering import meta_clustering # meta algorithms, to combine the different methods


from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score


c:\Users\bes3\AppData\Local\pypoetry\Cache\virtualenvs\timex-6QUUgBf4-py3.11\Lib\site-packages\tsfresh\__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
AKI_PATH = os.environ['AKI_PATH_NEW']
os.chdir(AKI_PATH)

ts_data = pd.read_parquet("data_preparation_DV_final_for_LCMM_19785AKIpatients_FU7-3650d_SW365d_TR90d_13032025.parquet")
ts_data = ts_data.assign(ID=ts_data.ID.astype('int64'))

TEST = True

In [3]:
# X = [n_ts, sz, dim]
# if there is no interpolation we assume that the time dimension is the second one
# i.e. we have at least 2 dimensions

ts_data = ts_data.rename(columns={"Time_since_index_AKI_days": "Time_days"})

In [4]:
ts_data = ts_data[['Time_days', 'eGFR_int', 'ID']]

In [5]:
if TEST:
    test_selection = random.choices(ts_data.ID.unique().tolist(), k=250)
else:
    test_selection = ts_data.ID.unique().tolist()

In [ ]:
clusters_8_timexCustom_outwards = model_based_clustering.clustering_mm(ts_data.loc[ts_data.ID.isin(test_selection)], 
                                                              'Time_days', 'eGFR_int', 'ID', 
                                    n_clusters=8, 
                                    spline_df=6, 
                                    spline_degree=3,
                                    how='normal', 
                                    num_samples=1500,
                                    direction='outwards',
                                    normalize=True)


In [ ]:
clusters_8_timexCustom_inwards = model_based_clustering.clustering_mm(ts_data.loc[ts_data.ID.isin(test_selection)], 
                                                              'Time_days', 'eGFR_int', 'ID', 
                                    n_clusters=8, 
                                    spline_df=6, 
                                    spline_degree=3,
                                    how='normal', 
                                    num_samples=1500,
                                    direction='inwards',
                                    normalize=True)


In [6]:
clusters_8_mm_slope = model_based_clustering.clustering_mm(ts_data.loc[ts_data.ID.isin(test_selection)], 
                                                                            'Time_days', 'eGFR_int', 'ID', 
                                    n_clusters=8, 
                                    spline_df=6, 
                                    spline_degree=3, 
                                    how='slope',
                                    num_samples=1500,
                                    normalize=True)

sample: 100%|██████████| 2000/2000 [00:12<00:00, 163.55it/s, 31 steps of size 1.30e-01. acc. prob=0.88]


(249, 6)


In [7]:
[len(c) for c in clusters_8_mm_slope]

[7, 1, 0, 5, 0, 3, 0, 1]

In [ ]:
clusters_8_mm_rec = model_based_clustering.clustering_mm(ts_data.loc[ts_data.ID.isin(test_selection)], 
                                                                            'Time_days', 'eGFR_int', 'ID', 
                                    n_clusters=8, 
                                    spline_df=6, 
                                    spline_degree=3, 
                                    how='random_effect_clustering',
                                    num_samples=1500,
                                    num_warmup=500,
                                    normalize=True)

In [ ]:
clusters_8_mm_ric = model_based_clustering.clustering_mm(ts_data.loc[ts_data.ID.isin(test_selection)], 
                                                                            'Time_days', 'eGFR_int', 'ID', 
                                    n_clusters=8, 
                                    spline_df=6, 
                                    spline_degree=3, 
                                    how='random_intercept_clustering',
                                    num_samples=1500,
                                    num_warmup=500,
                                    normalize=True)

In [ ]:
clusters_8_lmm = model_based_clustering.clustering_lmm(ts_data.loc[ts_data.ID.isin(test_selection)], 'Time_days', 'eGFR_int', 'ID', 
                                    n_clusters=8, 
                                    spline_df=8, 
                                    spline_degree=4, 
                                    num_samples=2000,
                                    num_warmup=500,
                                    discrete=False, # use Gumbel to make the sampling differentiable
                                    num_chains=1,
                                    vi = False,
                                    tau = 10,
                                    normalize=True, 
                                    adaptive_spline=True)

In [ ]:
[len(c) for c in clusters_8_lmm]

In [ ]:
clusters_8_lmm_vi = model_based_clustering.clustering_lmm(ts_data.loc[ts_data.ID.isin(test_selection)], 'Time_days', 'eGFR_int', 'ID', 
                                    n_clusters=8, 
                                    spline_df=5, 
                                    spline_degree=4, 
                                    num_steps=1000,
                                    discrete=False, # use Gumbel to make the sampling differentiable
                                    vi = True,
                                    tau_search = False,
                                    tau = 5,
                                    normalize=True, 
                                    adaptive_spline=True)

In [ ]:
[len(c) for c in clusters_8_lmm_vi]

In [ ]:
sum([len(c) for c in clusters_8_mm_slope])

In [ ]:
LOL = { 'lmm_vi': [i for i,c in enumerate(clusters_8_lmm_vi) for _c in c],
        'lmm':[i for i,c in enumerate(clusters_8_lmm) for _c in c],
        'mm_ric':[i for i,c in enumerate(clusters_8_mm_ric) for _c in c],
        'mm_rec':[i for i,c in enumerate(clusters_8_mm_rec) for _c in c], 
        'mm_slope':[i for i,c in enumerate(clusters_8_mm_slope) for _c in c],    
        'mm_inw':[i for i,c in enumerate(clusters_8_timexCustom_inwards) for _c in c], 
        'mm_out':[i for i,c in enumerate(clusters_8_timexCustom_outwards) for _c in c]
}

# get stats
res = []
for k,v1 in LOL.items():
    for l,v2 in LOL.items():
        res.append({
            'm1': k,
            'm2': l, 
            'ami': adjusted_mutual_info_score(v1,v2), 
            'ari': adjusted_rand_score(v1,v2)
            }) 
        


In [ ]:
pd.DataFrame(res)

In [ ]:
res_list = []
for clnum, idlist in enumerate(clusters_8_lmm):
    for id in idlist:
        res_list.append({
            'clusters_8_lmm': clnum, 
            'ID': id
        })

ts_data = pd.merge(ts_data, pd.DataFrame(res_list), how='left', on='ID')

#####################

res_list = []
for clnum, idlist in enumerate(clusters_8_lmm_vi):
    for id in idlist:
        res_list.append({
            'clusters_8_lmm_vi': clnum, 
            'ID': id
        })

ts_data = pd.merge(ts_data, pd.DataFrame(res_list), how='left', on='ID')



In [ ]:
ts_data = ts_data.fillna(-1)

In [ ]:
sns.lineplot(ts_data[ts_data.ID.isin(test_selection)], x='Time_days', y='eGFR_int', 
             hue='clusters_8_lmm_x', palette='tab10')

In [ ]:
num_plots_per_clusters = 100

cmap_ = plt.get_cmap('tab10', len(clusters_8))
cluster_colors = {c: cmap_(c) for c in range(len(clusters_8))}

fig, ax = plt.subplots(figsize=(18, 8))

for ck, cl in enumerate(clusters_8):
    for sid in random.choices(cl, k=num_plots_per_clusters):
        sub = ts_data[ts_data['ID'] == sid]
        ax.plot(sub['Time_days'], sub['eGFR_int'], lw=2, label=ck, color=cluster_colors[ck], alpha=0.1)
ax.set_title('Timeseries by Predicted Cluster')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper right', fontsize='small')